# Week B — reproduce a published figure

**Corrected version.** Four bugs from the first run are fixed:

1. `Ratio mod/base` gave only **23 testable sites of 2,056** — the stoichiometrically correct quantity is unusable for a low-stoichiometry modification. Now uses `Intensity` with explicit imputation.
2. Gene lookups used `Protein names` (descriptions) instead of `Gene names` (symbols), returning "not found" 14 times.
3. `res` was built with a fresh index while `clean` kept original row numbers — `KeyError: [0, 3, 4...] not in index`. Gene names are now carried into `res` at construction.
4. Imputation assigned a flat array to a boolean mask — `ValueError: other must be the same shape as self`. Now uses `.where()`.

**Target:** recover the ISGylation substrates reported in Pinto-Fernandez et al., Br J Cancer 124:817-830 (2021). Expect roughly 12 of 14.

## Step 1 — load

Re-downloads, so Week A's session need not still be running. FTP conversion folded in.

In [ ]:
import pandas as pd, numpy as np, requests, os
from scipy import stats

URL = ("https://ftp.pride.ebi.ac.uk/pride/data/archive/2022/02/"
       "PXD018299/HAP1_USP18KO_GlyGlyKSites.txt")
LOCAL = "HAP1_USP18KO_GlyGlyKSites.txt"

if not os.path.exists(LOCAL):
    with requests.get(URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(LOCAL, "wb") as fh:
            for chunk in r.iter_content(chunk_size=1 << 20):
                fh.write(chunk)

df = pd.read_csv(LOCAL, sep="\t", low_memory=False)
print(f"{len(df):,} rows, {len(df.columns)} columns")

## Step 2 — filter

Decoys and contaminants first. `LOC_THRESHOLD` is a **recorded decision** under invariant I16, not a constant — try 0 and 0.75 and see how much the answer moves.

In [ ]:
LOC_THRESHOLD = 0.75

clean = df[(df["Reverse"] != "+") & (df["Potential contaminant"] != "+")].copy()
clean = clean[clean["Localization prob"] >= LOC_THRESHOLD].copy()

print(f"start:                   {len(df):,}")
print(f"after loc >= {LOC_THRESHOLD}:       {len(clean):,}")

## Step 3 — choose the quantity

**Fix 1, and the most consequential decision in the notebook.**

`Ratio mod/base` divides modified-peptide intensity by unmodified protein signal, so it measures stoichiometry directly — theoretically the right quantity. But it exists only where *both* peptides are quantified in the same run, and for a low-stoichiometry modification like ISGylation that co-occurrence is rare. Requiring it in two replicates of both groups left **23 usable sites out of 2,056**, and all seven ADAR sites fell out.

`Intensity` is confounded by protein abundance — interferon induces most of these proteins — but it is populated. That is the trade the published analysis made, and reproducing the figure requires making it too.

Record both the choice and its consequence (I16).

In [ ]:
QUANTITY = "Intensity"   # 'Intensity' or 'Ratio mod/base'

KO_IFN = [f"{QUANTITY} KO_IFN_{i}" for i in (1, 2, 3)]
WT_IFN = [f"{QUANTITY} WT_IFN_{i}" for i in (1, 2, 3)]

missing = [c for c in KO_IFN + WT_IFN if c not in clean.columns]
if missing:
    print("NOT FOUND:", missing)
    print("\navailable:")
    for c in clean.columns:
        if c.startswith(QUANTITY):
            print(" ", repr(c))
else:
    print("all six columns found")

## Step 4 — log transform and impute

**Fix 4.** Assigning a flat array to a boolean mask raises a shape error; `.where()` takes a full-shape replacement frame instead.

Absence in mass spectrometry is ambiguous — the analyte may be genuinely missing, or present below detection. Perseus' default, and therefore the basis of the published figure, draws replacements from a normal distribution downshifted 1.8 SD below the observed mean with width 0.3 SD.

**These are generated numbers, not measurements.** The seed is mandatory: without it the same input gives a different answer, which breaks invariant I9. Under I15 the whole block must be recorded on the `Analysis`.

In [ ]:
IMPUTE_DOWNSHIFT = 1.8
IMPUTE_WIDTH = 0.3
IMPUTE_SEED = 0

vals = clean[KO_IFN + WT_IFN].apply(pd.to_numeric, errors="coerce").replace(0, np.nan)
log2 = np.log2(vals)

n_ko = log2[KO_IFN].notna().sum(axis=1)
n_wt = log2[WT_IFN].notna().sum(axis=1)
usable = (n_ko >= 2) | (n_wt >= 2)

obs = log2[usable]
mu, sd = np.nanmean(obs.values), np.nanstd(obs.values)

rng = np.random.default_rng(IMPUTE_SEED)
draws = rng.normal(mu - IMPUTE_DOWNSHIFT * sd, IMPUTE_WIDTH * sd, size=obs.shape)
imputed = obs.where(obs.notna(),
                    pd.DataFrame(draws, index=obs.index, columns=obs.columns))

n_imp = obs.isna().values.sum()
print(f"usable sites:   {usable.sum():,} of {len(clean):,}")
print(f"values imputed: {n_imp:,} of {obs.size:,} ({100*n_imp/obs.size:.1f}%)")
print(f"observed log2:  mean {mu:.2f}, sd {sd:.2f}")
print(f"imputed drawn:  mean {mu - IMPUTE_DOWNSHIFT*sd:.2f}, sd {IMPUTE_WIDTH*sd:.2f}")

sub = imputed
meta = clean[usable]

## Step 5 — the test

**Fix 2 and 3 are here.** `Gene names` (symbols) is carried into `res` at construction, using `.values` to strip the index so nothing can drift out of alignment later. The original used `Protein names`, which holds descriptions like "Vigilin" — no gene symbol ever matched.

Welch's *t*-test then Benjamini–Hochberg. Deliberately the naive version; Step 8 checks how much that matters.

In [ ]:
lfc = sub[KO_IFN].mean(axis=1) - sub[WT_IFN].mean(axis=1)
t_stat, p_raw = stats.ttest_ind(sub[KO_IFN], sub[WT_IFN], axis=1,
                                equal_var=False, nan_policy="omit")

def bh(p):
    p = np.asarray(p, dtype=float)
    ok = ~np.isnan(p)
    out = np.full_like(p, np.nan)
    q = p[ok]
    order = np.argsort(q)
    ranked = q[order]
    n = len(ranked)
    adj = ranked * n / (np.arange(n) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    res_ = np.empty(n)
    res_[order] = np.clip(adj, 0, 1)
    out[ok] = res_
    return out

res = pd.DataFrame({
    "protein": meta["Protein"].values,
    "proteins_all": meta["Proteins"].values,
    "gene": meta["Gene names"].values,
    "position": meta["Position"].values if "Position" in meta else None,
    "loc_prob": meta["Localization prob"].values,
    "log2fc": lfc.values,
    "p": np.asarray(p_raw, dtype=float),
})
res["adj_p"] = bh(res["p"])
res["n_candidate_proteins"] = res["proteins_all"].astype(str).str.count(";") + 1

sig = (res["adj_p"] < 0.05) & (res["log2fc"] > 1)
print(f"tested: {len(res):,}")
print(f"up in KO+IFN (adj p<0.05, log2FC>1): {sig.sum():,}")

## Step 6 — the check that matters

Exact match on gene symbols after splitting on semicolons — substring matching would let `OAS1` match `OASL`.

The `candidates` column is the number of proteins that site's peptide could have come from. Watch it: most published targets are not unambiguous.

In [ ]:
EXPECTED = ["ADAR", "EIF2AK2", "DDX58", "DDX60", "DHX58", "OAS1", "OAS2",
            "IFIH1", "STAT1", "PSMB9", "PSMB10", "PSMA7", "PSME2", "TAP1"]

genes = res["gene"].astype(str).str.upper()
found = 0

for g in EXPECTED:
    target = g.upper()
    mask = genes.str.split(";").apply(lambda lst: target in [x.strip() for x in lst])
    hits = res[mask]
    if len(hits) == 0:
        print(f"    {g:9s} not detected")
        continue
    best = hits.loc[hits["log2fc"].idxmax()]
    ok = best["adj_p"] < 0.05 and best["log2fc"] > 1
    found += ok
    print(f"{'OK  ' if ok else '    '}{g:9s} n_sites={len(hits):3d}  "
          f"best log2FC={best['log2fc']:+6.2f}  adj p={best['adj_p']:.2e}  "
          f"candidates={best['n_candidate_proteins']}")

print(f"\n{found} of {len(EXPECTED)} recovered as significantly up in KO+IFN")

## Step 7 — the volcano

Point size is inversely related to protein-assignment ambiguity (invariant I14). Small points rest on a razor pick. Most published volcanoes render every point as a confident single-protein label.

In [ ]:
import matplotlib.pyplot as plt

plot = res.dropna(subset=["log2fc", "adj_p"]).copy()
plot["neglog10q"] = -np.log10(plot["adj_p"].clip(lower=1e-300))
plot["unambiguous"] = plot["n_candidate_proteins"] == 1

fig, ax = plt.subplots(figsize=(8, 6))
for flag, size, alpha, label in [(False, 8, 0.25, "ambiguous protein"),
                                 (True, 26, 0.65, "unambiguous")]:
    s = plot[plot["unambiguous"] == flag]
    ax.scatter(s["log2fc"], s["neglog10q"], s=size, alpha=alpha,
               edgecolors="none", label=label)

ax.axhline(-np.log10(0.05), ls="--", lw=0.8, color="grey")
ax.axvline(1, ls="--", lw=0.8, color="grey")
ax.axvline(-1, ls="--", lw=0.8, color="grey")
ax.set_xlabel("log2 fold change  (KO+IFN vs WT+IFN)")
ax.set_ylabel("-log10 adjusted p")
ax.set_title(f"USP18-dependent GlyGly sites — PXD018299\n"
             f"{QUANTITY}, loc>={LOC_THRESHOLD}, downshift {IMPUTE_DOWNSHIFT} SD",
             fontsize=10)
ax.legend(frameon=False, loc="upper left")

top = plot[(plot["adj_p"] < 0.05) & (plot["log2fc"] > 1)].nlargest(12, "neglog10q")
for _, r in top.iterrows():
    label = str(r["gene"]).split(";")[0]
    ax.annotate(label, (r["log2fc"], r["neglog10q"]), fontsize=7, alpha=0.85)

plt.tight_layout()
plt.show()

## Step 8 — how much does the statistical test matter?

A crude stand-in for variance shrinkage: blend each site's variance with the dataset median. That is the intuition behind the moderated *t*-test — sites with implausibly small variance across three replicates get pulled toward typical.

The overlap is the number that matters. Broad agreement means the statistics layer is a refinement; substantial disagreement means it is load-bearing and `ARCHITECTURE.md` §4 is right to make it pluggable.

In [ ]:
d = sub[KO_IFN].mean(axis=1) - sub[WT_IFN].mean(axis=1)
var = (sub[KO_IFN].var(axis=1, ddof=1) + sub[WT_IFN].var(axis=1, ddof=1)) / 2
prior = np.nanmedian(var)

t_mod = d / np.sqrt(((var + prior) / 2) * (2 / 3))
res["adj_p_moderated"] = bh(2 * stats.t.sf(np.abs(t_mod), df=6))

a = set(res.index[(res["adj_p"] < 0.05) & (res["log2fc"] > 1)])
b = set(res.index[(res["adj_p_moderated"] < 0.05) & (res["log2fc"] > 1)])

print(f"naive Welch:     {len(a):,}")
print(f"shrunk variance: {len(b):,}")
print(f"in both:         {len(a & b):,}")
print(f"naive only:      {len(a - b):,}")
print(f"shrunk only:     {len(b - a):,}")

## Step 9 — the analysis record

Every choice that determined the result, written down. Invariants I15 and I16.

None of this is recoverable from a published methods section — which is the gap the platform exists to close.

In [ ]:
import json

analysis = {
    "dataset": "PXD018299",
    "file": LOCAL,
    "contrast": "KO_IFN_vs_WT_IFN",
    "quantity": QUANTITY,
    "localization_threshold": LOC_THRESHOLD,
    "filters_applied": ["reverse", "potential_contaminant"],
    "presence_rule": ">=2 replicates in either group",
    "imputation": {
        "method": "downshifted_normal",
        "downshift_sd": IMPUTE_DOWNSHIFT,
        "width_sd": IMPUTE_WIDTH,
        "seed": IMPUTE_SEED,
        "scope": "whole_matrix",
        "n_values_imputed": int(n_imp),
        "n_values_total": int(obs.size),
    },
    "test": "welch_t",
    "fdr_method": "BH",
    "protein_adjusted": "native" if QUANTITY == "Ratio mod/base" else "not_applied",
    "n_sites_tested": int(len(res)),
    "n_significant_up": int(sig.sum()),
    "n_expected_recovered": int(found),
    "n_expected_total": len(EXPECTED),
}

with open("analysis_PXD018299_KOIFN_vs_WTIFN.json", "w") as fh:
    json.dump(analysis, fh, indent=2)
print(json.dumps(analysis, indent=2))
print("\nDownload this alongside the curation record.")

## Step 10 — what to record

Into **Measured findings** in `ROADMAP.md`:

1. How many of the 14 came back, and which did not.
2. How much `LOC_THRESHOLD` changed it — run at 0 and 0.75.
3. How many recovered targets had ambiguous protein assignment. That is I14 quantified.
4. Whether the two statistical tests agreed.
5. What percentage of values were imputed.

**And the question the week exists to answer:** what was tedious? Remembering to filter contaminants, matching columns by hand, deciding about missing values, choosing which protein to label. Each one is a step the platform should do once, correctly, and record.

That list is your v0.1 specification, written from experience rather than design.

---

**Baseline for regression:** 12 of 14 recovered with `Intensity`, loc ≥ 0.75, downshift 1.8 SD, Welch + BH. Any future change that reduces this needs explaining.